# Hidden Key Employees — reproducible analysis (email-Eu-core-temporal)

Sahar Altaf and Nasir Hussain, Karachi Institute of Economics and Technology.

This notebook reproduces every number in the manuscript from the data snapshot stored in this repository:

1. clones the public repository and runs the brute-force verification;
2. uses the data snapshot in `data/raw/` (checked against `data/SHA256SUMS`); on the very first run, when no snapshot exists yet, it downloads the data from SNAP and creates the snapshot;
3. runs the main analysis, departure scenarios, sensitivity settings and robustness criteria;
4. records the run (`results/run_info.json`) and packs `data/` and `results/` into one zip for upload to GitHub.

**Runtime → Run all.** Output goes to Google Drive, `MyDrive/fuzzy-key-employees-output/`, so a disconnect loses nothing; re-running skips finished steps.

## 1. Setup

In [ ]:
REPO_URL = "https://github.com/nasirhussain92/fuzzy-key-employees.git"
N_RANDOM_RUNS = 100     # N_r, random runs for S6 (pre-specified)
SEED = 2026             # fixed seed for tie-breaking and random runs (pre-specified)

import os, sys, subprocess
if not os.path.exists("/content/repo"):
    subprocess.run(["git", "clone", "-q", REPO_URL, "/content/repo"], check=True)
os.chdir("/content/repo")
sys.path.insert(0, "/content/repo")
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()

from google.colab import drive
drive.mount("/content/drive")
EXPORT = "/content/drive/MyDrive/fuzzy-key-employees-output"
OUT = f"{EXPORT}/results"
DATA_OUT = f"{EXPORT}/data"
os.makedirs(OUT, exist_ok=True); os.makedirs(f"{DATA_OUT}/raw", exist_ok=True)
print("code commit:", COMMIT[:10], "| output folder:", EXPORT)

## 2. Verify the implementation
Must end with **ALL CHECKS PASSED**.

In [ ]:
r = subprocess.run([sys.executable, "verify.py"], capture_output=True, text=True)
print(r.stdout[-1500:])
assert "ALL CHECKS PASSED" in r.stdout, "verification failed - stop here"

## 3. Data snapshot
Uses `data/raw/` from the repository if present and checks every file against `data/SHA256SUMS`. If the snapshot is missing (first run only), downloads from SNAP, writes the checksums, and copies both to the output folder for committing.

In [ ]:
import hashlib, shutil, urllib.request
FILES = {
    "email-Eu-core-temporal.txt.gz": "https://snap.stanford.edu/data/email-Eu-core-temporal.txt.gz",
    "email-Eu-core-department-labels.txt.gz": "https://snap.stanford.edu/data/email-Eu-core-department-labels.txt.gz",
}
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

RAW = "data/raw"
have_snapshot = all(os.path.exists(f"{RAW}/{f}") for f in FILES) and os.path.exists("data/SHA256SUMS")
if have_snapshot:
    expected = dict(line.split()[::-1] for line in open("data/SHA256SUMS") if line.strip())
    for f in FILES:
        assert sha256(f"{RAW}/{f}") == expected[f], f"checksum mismatch for {f}"
    DATA_SOURCE = "repository snapshot (checksums verified)"
else:
    os.makedirs(RAW, exist_ok=True)
    for f, url in FILES.items():
        urllib.request.urlretrieve(url, f"{RAW}/{f}")
    with open("data/SHA256SUMS", "w") as out:
        for f in FILES:
            out.write(f"{sha256(f'{RAW}/{f}')}  {f}\n")
    for f in FILES:
        shutil.copy(f"{RAW}/{f}", f"{DATA_OUT}/raw/{f}")
    shutil.copy("data/SHA256SUMS", f"{DATA_OUT}/SHA256SUMS")
    DATA_SOURCE = "downloaded from SNAP on first run; snapshot created"
print(DATA_SOURCE)
print(open("data/SHA256SUMS").read())

import math, json, itertools, datetime, platform
import numpy as np, pandas as pd, networkx as nx, scipy
from scipy.stats import kendalltau
import fuzzy_hrm as fh

events = fh.load_eu_core_temporal(f"{RAW}/email-Eu-core-temporal.txt.gz")
dept = fh.load_eu_core_departments(f"{RAW}/email-Eu-core-department-labels.txt.gz")
nodes = set(events.sender) | set(events.receiver)
print(f"{len(events):,} email records, {len(nodes)} people, {len(dept)} department labels")
print(f"people with a department label: {len(nodes & set(dept))} of {len(nodes)}")

## 4. Main analysis (λ = λ₀, η = 0)

In [ ]:
def build(eta_on, lam_factor):
    eta = fh.eta0(events) if eta_on else 0.0
    w = fh.link_weights(events, eta=eta)
    sigma, mu = fh.build_fuzzy_graph(w, lam=lam_factor * fh.lambda0(w), vertices=nodes)
    V = sorted(sigma)
    return V, mu, fh.sorted_arcs(mu)

V, mu, arcs = build(False, 1.0)
ks, T = fh.key_person_scores(V, arcs)
fuzzy_cut, hidden, _ = fh.hidden_key_employees(V, mu, ks, p=0.05)
per = pd.DataFrame({"employee": V, "department": [dept.get(v) for v in V],
                    "KS": [ks[v] for v in V], "T": [T[v] for v in V]})
per["KS_rank"] = per.KS.rank(ascending=False, method="min").astype(int)
per["fuzzy_cutvertex"] = per.employee.isin(fuzzy_cut)
per["hidden"] = per.employee.isin(hidden)
summary = {"employees": len(V), "arcs": len(mu), "distinct_levels": len(set(mu.values())),
           "NCI": fh.nci(V, arcs), "fuzzy_cutvertices": len(fuzzy_cut), "hidden": len(hidden)}
for p in (0.01, 0.05, 0.10):
    _, _, material = fh.hidden_key_employees(V, mu, ks, p=p)
    per[f"material_p{int(p*100)}"] = per.employee.isin(material)
    summary[f"material_p{int(p*100)}"] = len(material)
bridges = fh.fuzzy_bridges(V, mu, arcs)
summary["fuzzy_bridges"] = len(bridges)
summary["inter_department_bridges"] = int(sum(dept.get(a) != dept.get(b) for a, b in bridges))
per.sort_values("KS", ascending=False).to_csv(f"{OUT}/main_employee_scores.csv", index=False)
pd.DataFrame(bridges, columns=["u", "v"]).to_csv(f"{OUT}/main_fuzzy_bridges.csv", index=False)
json.dump(summary, open(f"{OUT}/main_summary.json", "w"), indent=2,
          default=lambda x: x.item() if hasattr(x, "item") else str(x))
summary

## 5. Fuzzy vs other rankings (Section 4.6)

In [ ]:
rows = []
top_ks = fh.rank(ks, SEED)
for name, sc in fh.strategy_scores(V, mu, ks).items():
    if name.startswith("S1"):
        continue
    top_s = fh.rank(sc, SEED)
    row = {"strategy": name, "kendall_tau_vs_KS": kendalltau([ks[v] for v in V], [sc[v] for v in V]).statistic}
    for k in (10, 20, 50):
        row[f"top{k}_shared"] = len(set(top_ks[:k]) & set(top_s[:k]))
    rows.append(row)
comparison = pd.DataFrame(rows)
comparison.to_csv(f"{OUT}/main_rank_comparison.csv", index=False)
comparison

## 6. Departure scenarios and sensitivity settings (Sections 4.5–4.6)
One-at-a-time settings: main (λ₀, η = 0), λ = 0.5 λ₀, λ = 2 λ₀ (η = 0), η = η₀ (λ₀); waves 1%, 5%, 10%. Each finished run is saved at once and skipped on re-run.

In [ ]:
SETTINGS = {"main": (False, 1.0), "lambda_x0.5": (False, 0.5),
            "lambda_x2": (False, 2.0), "eta0": (True, 1.0)}
results, allres = {}, []
for tag, (eta_on, lam_f) in SETTINGS.items():
    Vg, mug, arcsg = build(eta_on, lam_f)
    ksg, _ = fh.key_person_scores(Vg, arcsg)
    results[tag] = {"ks": ksg, "scen": {}}
    for wave in (0.01, 0.05, 0.10):
        path = f"{OUT}/scenarios_{tag}_K{int(wave*100)}.csv"
        if os.path.exists(path):
            res = pd.read_csv(path)
        else:
            res = fh.run_scenarios(Vg, mug, ksg, wave=wave, n_random_runs=N_RANDOM_RUNS, seed=SEED)
            res.to_csv(path, index=False)
        results[tag]["scen"][wave] = res
        allres.append(res.assign(setting=tag, wave=wave))
    print("done", tag)
pd.concat(allres, ignore_index=True).to_csv(f"{OUT}/scenarios_all.csv", index=False)
results["main"]["scen"][0.05]

## 7. Robustness criteria (pre-specified)
Three criteria, reported separately: S1 first by L_S; Kendall's τ ≥ 0.70 over employees with KS > 0 in either setting; top-5% Jaccard ≥ 0.50. Failures are reported with direction and size.

In [ ]:
rob = []
for tag in SETTINGS:
    if tag == "main":
        continue
    for wave in (0.01, 0.05, 0.10):
        r = fh.robustness({"ks": results["main"]["ks"], "scenarios": results["main"]["scen"][wave]},
                          {"ks": results[tag]["ks"], "scenarios": results[tag]["scen"][wave]}, p=0.05)
        rob.append({"setting": tag, "wave": wave, **r})
rob = pd.DataFrame(rob)
rob.to_csv(f"{OUT}/robustness.csv", index=False)
for wave in (0.01, 0.05, 0.10):
    order = results["main"]["scen"][wave].sort_values("L_S (mean R)", ascending=False).strategy.tolist()
    print(f"main setting, wave {int(wave*100)}%: S1 rank = {order.index('S1 Fuzzy KS') + 1}")
print("Criterion 1  S1 first in every setting and wave:", bool(rob.S1_first.all()))
print("Criterion 2  tau >= 0.70 in every setting:     ", bool(rob.tau_pass.all()))
print("Criterion 3  Jaccard >= 0.50 in every setting: ", bool(rob.jaccard_pass.all()))
rob.round(3)

## 8. Record the run and pack everything for GitHub
Writes `results/run_info.json` and creates `upload_to_github.zip` in the output folder. Extract it and upload its `data/` and `results/` folders to the repository root.

In [ ]:
run_info = {
    "date_utc": datetime.datetime.utcnow().isoformat(timespec="seconds") + "Z",
    "code_commit": COMMIT,
    "data_source": DATA_SOURCE,
    "data_sha256": dict(line.split()[::-1] for line in open("data/SHA256SUMS") if line.strip()),
    "parameters": {"lambda": "lambda0 = ln2 / median positive w_uv", "eta_main": 0,
                   "eta0": "2 ln2 / H", "departure_waves": [0.01, 0.05, 0.10],
                   "materiality_p": [0.01, 0.05, 0.10], "n_random_runs": N_RANDOM_RUNS, "seed": SEED,
                   "robustness": {"tau_min": 0.70, "jaccard_min": 0.50, "top_group_p": 0.05}},
    "python": platform.python_version(),
    "packages": {"numpy": np.__version__, "pandas": pd.__version__,
                 "networkx": nx.__version__, "scipy": scipy.__version__},
}
json.dump(run_info, open(f"{OUT}/run_info.json", "w"), indent=2)
with open(f"{OUT}/requirements-lock.txt", "w") as f:
    for k, v in run_info["packages"].items():
        f.write(f"{k}=={v}\n")
shutil.make_archive(f"{EXPORT}/upload_to_github", "zip", EXPORT, ".")
print("created", f"{EXPORT}/upload_to_github.zip")
print(json.dumps(run_info, indent=2))

## Not yet included
* **RQ2 bridge share** (share of inter-department pairs whose CONN falls when a bridge is removed): bridges are saved in `main_fuzzy_bridges.csv`; the share calculation is the next code step.
* **Enron** (RQ1 hierarchy alignment): needs a parser from raw emails to a small `sender, receiver, time` table, which will be committed to `data/` in place of the 420 MB corpus.